### Imports and logger setup

In [0]:
import sys
sys.path.append("/Workspace/claim-denial-prevention")

from logger import (
    get_logger, log_start, log_end,
    log_step, log_success, log_warning, log_error
)
from error_codes import ErrorCodes

logger = get_logger("eda")
log_start(logger, "eda")

### Configuration

In [0]:
try:
    BASE_PATH = "/Volumes/workspace/default/myvol"

    BRONZE_PATHS = {
        "claims":    f"{BASE_PATH}/bronze/claims/",
        "providers": f"{BASE_PATH}/bronze/providers/",
        "diagnosis": f"{BASE_PATH}/bronze/diagnosis/",
        "cost":      f"{BASE_PATH}/bronze/cost/",
    }

    CLAIMS_COLUMNS = [
        "claim_id", "patient_id", "provider_id",
        "diagnosis_code", "procedure_code",
        "billed_amount", "claim_date", "denial_flag"
    ]

    log_success(logger, "Configuration loaded successfully")
    logger.info(f"Base path: {BASE_PATH}")

except Exception as e:
    log_error(
        logger,
        ErrorCodes.INF_001,
        ErrorCodes.get_message(ErrorCodes.INF_001),
        e
    )
    raise

### Load all 4 Bronze tables

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

log_step(logger, 1, "Loading all 4 Bronze Delta tables")

dataframes = {}

for name, path in BRONZE_PATHS.items():
    try:
        logger.info(f"Loading '{name}' from: {path}")
        df          = spark.read.format("delta").load(path)
        row_count   = df.count()
        col_count   = len(df.columns)
        dataframes[name] = df
        log_success(logger, f"[{name}] {row_count} rows × {col_count} columns")

    except Exception as e:
        log_error(
            logger,
            ErrorCodes.EDA_001,
            f"Failed to load bronze table '{name}' from {path}",
            e
        )
        raise RuntimeError(
            ErrorCodes.format_error(
                ErrorCodes.EDA_001,
                f"Cannot proceed without '{name}' table. Ensure Week 1 bronze ingestion is complete."
            )
        )

# Assign named variables
df_claims    = dataframes["claims"]
df_providers = dataframes["providers"]
df_diagnosis = dataframes["diagnosis"]
df_cost      = dataframes["cost"]

print("\n" + "=" * 50)
print("BRONZE TABLE SHAPE SUMMARY")
print("=" * 50)
for name, df in dataframes.items():
    print(f"  {name:12s} : {df.count():>6} rows × {len(df.columns):>2} columns")
print("=" * 50)

### Null value analysis on claims table

In [0]:
log_step(logger, 2, "Null Value Analysis — Claims Table")

try:
    total_claims = df_claims.count()
    logger.info(f"Total claims for null analysis: {total_claims}")

    if total_claims == 0:
        log_error(
            logger,
            ErrorCodes.BRZ_003,
            ErrorCodes.get_message(ErrorCodes.BRZ_003)
        )
        raise ValueError(ErrorCodes.format_error(ErrorCodes.BRZ_003))

    print("\n" + "=" * 60)
    print("CLAIMS: NULL VALUE ANALYSIS")
    print("=" * 60)
    print(f"  {'Column':<22} {'Null Count':>10} {'Null %':>8}  {'Status'}")
    print("-" * 60)

    high_null_columns = []

    for col_name in CLAIMS_COLUMNS:
        try:
            null_count = df_claims.filter(F.col(col_name).isNull()).count()
            pct        = round(null_count / total_claims * 100, 2)
            status     = "⚠️  HIGH" if pct > 5 else "✅ OK"

            print(f"  {col_name:<22} {null_count:>10} {pct:>7.2f}%  {status}")

            if pct > 5:
                high_null_columns.append(col_name)
                log_warning(
                    logger,
                    ErrorCodes.EDA_002,
                    f"High null rate on '{col_name}': {null_count} nulls ({pct}%) "
                    f"— imputation will be applied in Silver layer"
                )

        except Exception as col_err:
            log_error(
                logger,
                ErrorCodes.EDA_002,
                f"Failed to analyze null count for column '{col_name}'",
                col_err
            )

    print("=" * 60)

    if high_null_columns:
        logger.info(f"Columns requiring imputation in Silver: {high_null_columns}")
    else:
        log_success(logger, "No columns with high null rates (>5%) detected")

    log_success(logger, "Null analysis completed")

except ValueError:
    raise
except Exception as e:
    log_error(
        logger,
        ErrorCodes.EDA_002,
        ErrorCodes.get_message(ErrorCodes.EDA_002),
        e
    )
    raise

### Duplicate claim_id check

In [0]:
log_step(logger, 3, "Duplicate claim_id Check")

try:
    total  = df_claims.count()
    unique = df_claims.select("claim_id").distinct().count()
    dupes  = total - unique

    print("\n" + "=" * 50)
    print("DUPLICATE CLAIM ID CHECK")
    print("=" * 50)
    print(f"  Total claims      : {total}")
    print(f"  Unique claim_ids  : {unique}")
    print(f"  Duplicates found  : {dupes}")
    print("=" * 50)

    if dupes > 0:
        log_warning(
            logger,
            ErrorCodes.EDA_003,
            f"{dupes} duplicate claim_ids found — "
            f"will be resolved in Silver layer using window deduplication"
        )

        # Show sample duplicates for investigation
        logger.info("Sample duplicate records:")
        try:
            window_spec = Window.partitionBy("claim_id").orderBy("claim_id")
            df_claims \
                .withColumn("dupe_count", F.count("claim_id").over(window_spec)) \
                .filter(F.col("dupe_count") > 1) \
                .select("claim_id", "patient_id", "_ingestion_timestamp") \
                .show(5, truncate=False)
        except Exception as sample_err:
            log_warning(
                logger,
                ErrorCodes.EDA_003,
                f"Could not display sample duplicates: {str(sample_err)}"
            )
    else:
        log_success(logger, "No duplicate claim_ids found")

    log_success(logger, "Duplicate check completed")

except Exception as e:
    log_error(
        logger,
        ErrorCodes.EDA_003,
        ErrorCodes.get_message(ErrorCodes.EDA_003),
        e
    )
    raise

### Billed amount distribution

In [0]:
log_step(logger, 4, "Billed Amount Analysis")

try:
    print("\n" + "=" * 50)
    print("BILLED AMOUNT STATS")
    print("=" * 50)
    df_claims.select("billed_amount").describe().show()

    # Invalid amounts
    invalid_amounts = df_claims.filter(
        F.col("billed_amount").isNull() |
        (F.col("billed_amount") <= 0)
    ).count()

    print(f"  Invalid amounts (null or ≤0): {invalid_amounts}")

    if invalid_amounts > 0:
        log_warning(
            logger,
            ErrorCodes.EDA_004,
            f"{invalid_amounts} claims have invalid billed_amount (null or ≤0) "
            f"— will be set to 0.0 in Silver layer"
        )
    else:
        log_success(logger, "All billed amounts are valid (non-null and >0)")

    # Outlier detection
    try:
        avg_amount = df_claims.agg(F.avg("billed_amount")).collect()[0][0]
        if avg_amount is not None:
            outliers = df_claims.filter(
                F.col("billed_amount") > avg_amount * 3
            ).count()
            print(f"  Average billed amount  : ${round(avg_amount, 2)}")
            print(f"  Outliers (>3x average) : {outliers} claims")
            logger.info(f"Average billed amount: ${round(avg_amount, 2)}")
            logger.info(f"Outliers (>3x average): {outliers} claims")

            if outliers > 0:
                log_warning(
                    logger,
                    ErrorCodes.EDA_004,
                    f"{outliers} claims have billed_amount >3x the average "
                    f"(${round(avg_amount, 2)}) — potential overbilling, review in EDA dashboard"
                )
        else:
            log_warning(
                logger,
                ErrorCodes.EDA_004,
                "Could not compute average billed amount — column may be all nulls"
            )
    except Exception as outlier_err:
        log_warning(
            logger,
            ErrorCodes.EDA_004,
            f"Outlier detection skipped: {str(outlier_err)}"
        )

    print("=" * 50)
    log_success(logger, "Billed amount analysis completed")

except Exception as e:
    log_error(
        logger,
        ErrorCodes.EDA_004,
        ErrorCodes.get_message(ErrorCodes.EDA_004),
        e
    )
    raise

### Denial Flag Distribution

In [0]:
# Cell 7 — Denial flag distribution
log_step(logger, 5, "Denial Flag Distribution")

try:
    total_claims  = df_claims.count()
    denied_claims = df_claims.filter(F.col("denial_flag") == 1).count()
    denial_rate   = round(denied_claims / total_claims * 100, 2) if total_claims > 0 else 0.0

    print("\n" + "=" * 50)
    print("DENIAL FLAG DISTRIBUTION")
    print("=" * 50)
    df_claims.groupBy("denial_flag").count().orderBy("denial_flag").show()
    print(f"  Overall denial rate: {denial_rate}%")
    print("=" * 50)

    logger.info(f"Denial rate: {denial_rate}% ({denied_claims}/{total_claims})")

    if denial_rate < 5:
        log_warning(
            logger,
            ErrorCodes.EDA_005,
            f"Very low denial rate ({denial_rate}%) — dataset may be heavily imbalanced. "
            f"Consider class weighting in ML model (Week 3)"
        )
    elif denial_rate > 50:
        log_warning(
            logger,
            ErrorCodes.EDA_005,
            f"High denial rate ({denial_rate}%) — verify denial_flag values are correct"
        )
    else:
        log_success(logger, f"Denial rate looks reasonable: {denial_rate}%")

    # Check for unexpected denial_flag values
    valid_flags = df_claims.filter(
        F.col("denial_flag").isin([0, 1])
    ).count()
    invalid_flags = total_claims - valid_flags
    if invalid_flags > 0:
        log_warning(
            logger,
            ErrorCodes.EDA_005,
            f"{invalid_flags} claims have unexpected denial_flag values (not 0 or 1)"
        )
    else:
        log_success(logger, "All denial_flag values are valid (0 or 1)")

    log_success(logger, "Denial flag analysis completed")

except Exception as e:
    log_error(
        logger,
        ErrorCodes.EDA_005,
        ErrorCodes.get_message(ErrorCodes.EDA_005),
        e
    )
    raise

### Diagnosis code format check 

In [0]:
# Cell 8 — Diagnosis code format check
log_step(logger, 6, "Diagnosis Code Format Check")

try:
    print("\n" + "=" * 50)
    print("DIAGNOSIS CODE FORMAT CHECK")
    print("=" * 50)

    # Check for lowercase letters
    lowercase_codes = df_claims.filter(
        F.col("diagnosis_code").rlike("[a-z]")
    ).count()

    # Check for leading/trailing whitespace
    whitespace_codes = df_claims.filter(
        F.col("diagnosis_code") != F.trim(F.col("diagnosis_code"))
    ).count()

    # Check for nulls
    null_codes = df_claims.filter(
        F.col("diagnosis_code").isNull()
    ).count()

    total_format_issues = lowercase_codes + whitespace_codes

    print(f"  Lowercase codes         : {lowercase_codes}")
    print(f"  Whitespace codes        : {whitespace_codes}")
    print(f"  Null codes              : {null_codes}")
    print(f"  Total format issues     : {total_format_issues}")
    print("=" * 50)

    if total_format_issues > 0:
        log_warning(
            logger,
            ErrorCodes.EDA_006,
            f"{total_format_issues} diagnosis codes have format issues "
            f"(lowercase: {lowercase_codes}, whitespace: {whitespace_codes}) "
            f"— will be standardized in Silver layer"
        )

        # Show sample bad codes
        logger.info("Sample diagnosis codes with format issues:")
        df_claims.filter(
            F.col("diagnosis_code").rlike("[a-z]") |
            (F.col("diagnosis_code") != F.trim(F.col("diagnosis_code")))
        ).select("claim_id", "diagnosis_code").show(5, truncate=False)

    else:
        log_success(logger, "All diagnosis codes are properly formatted")

    if null_codes > 0:
        log_warning(
            logger,
            ErrorCodes.EDA_006,
            f"{null_codes} null diagnosis codes found — will be replaced with 'UNKNOWN' in Silver"
        )

    log_success(logger, "Diagnosis code format check completed")

except Exception as e:
    log_error(
        logger,
        ErrorCodes.EDA_006,
        ErrorCodes.get_message(ErrorCodes.EDA_006),
        e
    )
    raise

### Join feasibility check

In [0]:
# Cell 9 — Join feasibility check
log_step(logger, 7, "Join Feasibility Check")

try:
    print("\n" + "=" * 60)
    print("JOIN FEASIBILITY CHECK")
    print("=" * 60)

    # Provider match
    try:
        claims_providers   = df_claims.select("provider_id").distinct()
        known_providers    = df_providers.select("provider_id").distinct()
        unmatched_prov     = claims_providers.subtract(known_providers).count()
        total_claim_provs  = claims_providers.count()
        match_rate_prov    = round((total_claim_provs - unmatched_prov) / total_claim_provs * 100, 2)

        print(f"  Providers  — Unmatched: {unmatched_prov:>5} | Match rate: {match_rate_prov}%")
        logger.info(f"Provider match rate: {match_rate_prov}% ({unmatched_prov} unmatched)")

        if unmatched_prov > 0:
            log_warning(
                logger,
                ErrorCodes.EDA_007,
                f"{unmatched_prov} provider_ids in claims have no match in providers table "
                f"— these will show UNKNOWN specialty/location after left join"
            )
    except Exception as prov_err:
        log_error(
            logger,
            ErrorCodes.EDA_007,
            "Provider join feasibility check failed",
            prov_err
        )

    # Diagnosis match
    try:
        claims_diag    = df_claims.select("diagnosis_code").distinct()
        known_diag     = df_diagnosis.select("diagnosis_code").distinct()
        unmatched_diag = claims_diag.subtract(known_diag).count()
        total_diag     = claims_diag.count()
        match_rate_diag = round((total_diag - unmatched_diag) / total_diag * 100, 2)

        print(f"  Diagnosis  — Unmatched: {unmatched_diag:>5} | Match rate: {match_rate_diag}%")
        logger.info(f"Diagnosis match rate: {match_rate_diag}% ({unmatched_diag} unmatched)")

        if unmatched_diag > 0:
            log_warning(
                logger,
                ErrorCodes.EDA_007,
                f"{unmatched_diag} diagnosis_codes in claims have no match in diagnosis table "
                f"— these will show UNKNOWN category/severity after left join"
            )
    except Exception as diag_err:
        log_error(
            logger,
            ErrorCodes.EDA_007,
            "Diagnosis join feasibility check failed",
            diag_err
        )

    # Procedure/cost match
    try:
        claims_proc     = df_claims.select("procedure_code").distinct()
        known_proc      = df_cost.select("procedure_code").distinct()
        unmatched_proc  = claims_proc.subtract(known_proc).count()
        total_proc      = claims_proc.count()
        match_rate_proc = round((total_proc - unmatched_proc) / total_proc * 100, 2)

        print(f"  Procedures — Unmatched: {unmatched_proc:>5} | Match rate: {match_rate_proc}%")
        logger.info(f"Procedure match rate: {match_rate_proc}% ({unmatched_proc} unmatched)")

        if unmatched_proc > 0:
            log_warning(
                logger,
                ErrorCodes.EDA_007,
                f"{unmatched_proc} procedure_codes in claims have no match in cost table "
                f"— these will show 0.0 average_cost/expected_cost after left join"
            )
    except Exception as proc_err:
        log_error(
            logger,
            ErrorCodes.EDA_007,
            "Procedure join feasibility check failed",
            proc_err
        )

    print("=" * 60)
    log_success(logger, "Join feasibility check completed")

except Exception as e:
    log_error(
        logger,
        ErrorCodes.EDA_007,
        ErrorCodes.get_message(ErrorCodes.EDA_007),
        e
    )
    raise

## EDA Summary Report

In [0]:
# Cell 10 — EDA Summary Report
log_step(logger, 8, "EDA Summary Report")

try:
    total_claims  = df_claims.count()
    denied_claims = df_claims.filter(F.col("denial_flag") == 1).count()
    denial_rate   = round(denied_claims / total_claims * 100, 2) if total_claims > 0 else 0

    print("\n" + "=" * 60)
    print("WEEK 2 EDA BRONZE — SUMMARY REPORT")
    print("=" * 60)
    print(f"  Total Claims             : {total_claims}")
    print(f"  Denied Claims            : {denied_claims}")
    print(f"  Overall Denial Rate      : {denial_rate}%")
    print(f"  Unique Providers         : {df_claims.select('provider_id').distinct().count()}")
    print(f"  Unique Diagnosis Codes   : {df_claims.select('diagnosis_code').distinct().count()}")
    print(f"  Unique Procedure Codes   : {df_claims.select('procedure_code').distinct().count()}")
    print("=" * 60)
    print("\n  ACTION ITEMS FOR SILVER LAYER:")
    print("  1. Remove duplicate claim_ids (if any found above)")
    print("  2. Impute null diagnosis_code / procedure_code → 'UNKNOWN'")
    print("  3. Set invalid billed_amount → 0.0")
    print("  4. Standardize codes → UPPERCASE + trim whitespace")
    print("  5. Left join providers, diagnosis, cost tables")
    print("  6. Add claim_age_days and missing code flag columns")
    print("=" * 60)

    log_success(logger, "EDA summary report generated")

except Exception as e:
    log_error(
        logger,
        ErrorCodes.EDA_001,
        "EDA summary report generation failed",
        e
    )
    raise

finally:
    log_end(logger, "eda")